# NER Model Training for Calendar Bot

NER на базе `bert-base-multilingual-cased` для извлечения сущностей из текстовых запросов:
- DATE (дата)
- TIME (время)
- LOC (место)
- TITLE (название события)
- USER (участники)
- URL (ссылки)

## Параметры обучения
- Learning rate: 2e-5
- Epochs: 4
- Batch size: 16
- Validation split: 10%


## 1. Imports & Setup


In [ ]:
pip install seqeval

In [ ]:
import json
import os
import numpy as np
import torch
from pathlib import Path
from typing import Dict, List, Any

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)
from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

FORCE_DEVICE = "cuda"

if FORCE_DEVICE == "auto":
    if torch.cuda.is_available():
        DEVICE = torch.device("cuda")
        print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
    elif torch.backends.mps.is_available():
        DEVICE = torch.device("mps")
        print("Using Apple Silicon MPS")
    else:
        DEVICE = torch.device("cpu")
        print("Using CPU")
elif FORCE_DEVICE == "cuda" and torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
elif FORCE_DEVICE == "mps" and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Using Apple Silicon MPS")
else:
    DEVICE = torch.device("cuda")
    print("Using CPU (forced or fallback)")

print(f"PyTorch version: {torch.__version__}")
print(f"Selected device: {DEVICE}")


Варианты: "auto", "cpu", "mps", "cuda"
 - "auto" - автоматический выбор лучшего доступного
 - "cpu"  - принудительно CPU
 - "mps"  - Apple Silicon GPU
 - "cuda" - GPU-шка, если есть

## 2. Configuration


In [ ]:
PROJECT_ROOT = Path("")
DATA_PATH = PROJECT_ROOT / "data" / "dataset.jsonl"
OUTPUT_DIR = Path("./checkpoints")
FINAL_MODEL_DIR = Path("./trained_model")

MODEL_NAME = "DeepPavlov/rubert-base-cased"

LEARNING_RATE = 2e-5
NUM_EPOCHS = 4
VALIDATION_SPLIT = 0.1
MAX_LENGTH = 128
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

if DEVICE.type == "cuda":
    BATCH_SIZE = 16
    GRADIENT_ACCUMULATION_STEPS = 1
    print("Config for CUDA GPU - full batch size")
elif DEVICE.type == "mps":
    BATCH_SIZE = 2
    GRADIENT_ACCUMULATION_STEPS = 8
    print("Config for Apple Silicon MPS - reduced batch size")
else:
    BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 2
    print("Config for CPU")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Batch size: {BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")


## 3. Define Label Mappings

BIO-схема тегов для 6 типов сущностей + O (outside)


In [ ]:
LABEL_LIST = [
    "O",       
    "B-DATE",  
    "I-DATE",  
    "B-TIME",
    "I-TIME",   
    "B-LOC",
    "I-LOC", 
    "B-TITLE", 
    "I-TITLE", 
    "B-USER", 
    "I-USER",  
    "B-URL",   
    "I-URL", 
]

#label <-> id
label2id = {label: idx for idx, label in enumerate(LABEL_LIST)}
id2label = {idx: label for idx, label in enumerate(LABEL_LIST)}

NUM_LABELS = len(LABEL_LIST)
print(f"Number of labels: {NUM_LABELS}")
print(f"Labels: {LABEL_LIST}")

## 4. Load Dataset


In [ ]:

##### WARNING -- путь указан для kaggle, в проекте надо поменять репо ###
DATA_PATH = '/kaggle/input/realy-final-data/dataset_full.jsonl'

def load_jsonl(file_path: Path) -> List[Dict[str, Any]]:
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

raw_data = load_jsonl(DATA_PATH)
print(f"Loaded {len(raw_data)} examples")

print("\nПример данных:")
print(json.dumps(raw_data[0], ensure_ascii=False, indent=2))

In [ ]:
def convert_tags_to_ids(examples: List[Dict]) -> List[Dict]:
    """Преобразование строковых тегов в числовые ID"""
    converted = []
    for ex in examples:
        tag_ids = [label2id.get(tag, 0) for tag in ex["ner_tags"]]
        converted.append({
            "id": ex["id"],
            "lang": ex["lang"],
            "tokens": ex["tokens"],
            "ner_tags": tag_ids,
            "text": ex["text"],
        })
    return converted

processed_data = convert_tags_to_ids(raw_data)
print(f"Converted {len(processed_data)} examples")
print(f"\nПример конвертированных тегов:")
print(f"Tokens: {processed_data[0]['tokens'][:10]}...")
print(f"Tags: {processed_data[0]['ner_tags'][:10]}...")


## 5. Train/Validation Split


In [ ]:
train_data, val_data = train_test_split(
    processed_data,
    test_size=VALIDATION_SPLIT,
    random_state=SEED,
)

print(f"Train size: {len(train_data)}")
print(f"Validation size: {len(val_data)}")

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
})

print(f"\nDataset structure:")
print(dataset)


## 6. Tokenization with Label Alignment

При использовании BERT токенизатора слова могут разбиваться на подтокены (subwords).
Нужно правильно выровнять метки для каждого подтокена

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")


In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True,
        padding=False,
    )
    
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Начало нового слова -> берем метку как есть
                label_ids.append(label[word_idx])
            else:
                # --- ИСПРАВЛЕНИЕ ЗДЕСЬ ---
                # Это хвост (subword). Мы НЕ игнорируем его (-100).
                # Мы копируем метку, но меняем B- (Begin) на I- (Inside).
                original_label_id = label[word_idx]
                
                # Если исходная метка была игнорируемой (вдруг), оставляем
                if original_label_id == -100:
                    label_ids.append(-100)
                else:
                    label_name = id2label[original_label_id]
                    # Превращаем B-TAG в I-TAG для хвоста
                    if label_name.startswith("B-"):
                        label_name = "I-" + label_name[2:]
                        label_ids.append(label2id[label_name])
                    else:
                        # Если было I-TAG или O, оставляем как есть
                        label_ids.append(original_label_id)
                # -------------------------
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs



In [ ]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing dataset",
)

print("Tokenized dataset:")
print(tokenized_dataset)

print("\nПример токенизированных данных:")
example = tokenized_dataset["train"][0]
print(f"Input IDs length: {len(example['input_ids'])}")
print(f"Labels length: {len(example['labels'])}")


## 7. Initialize Model


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

if DEVICE.type != "cpu":
    model = model.to(DEVICE)

print(f"Model loaded: {MODEL_NAME}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"Model device: {next(model.parameters()).device}")


## 8. Metrics


In [ ]:
def compute_metrics(eval_preds):
    """
    Считаем метрики с помощью seqeval
    """
    predictions, labels = eval_preds
    predictions = np.argmax(predictions, axis=2)
    
    true_predictions = []
    true_labels = []
    
    for prediction, label in zip(predictions, labels):
        true_pred = []
        true_lab = []
        
        for p, l in zip(prediction, label):
            if l != -100:  # Игнорируем padding и специальные токены
                true_pred.append(id2label[p])
                true_lab.append(id2label[l])
        
        true_predictions.append(true_pred)
        true_labels.append(true_lab)
    
    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }


## 9. Training Setup


In [ ]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)


In [ ]:
use_cuda = (DEVICE.type == "cuda")
use_mps = (DEVICE.type == "mps")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    optim="adamw_torch",
    
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=50,
    report_to="none",
    
    use_mps_device=use_mps,
    fp16=use_cuda,
    dataloader_pin_memory=use_cuda,
    no_cuda=(not use_cuda),
    
    seed=SEED,
    save_total_limit=2,
    remove_unused_columns=True,
)

print("Training arguments configured")
print(f"  - Device: {DEVICE}")
print(f"  - Epochs: {NUM_EPOCHS}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  - Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  - Learning rate: {LEARNING_RATE}")
print(f"  - FP16: {use_cuda}")


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Trainer initialized")


## 10. Train


In [ ]:
import gc
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

print("Starting training...")
print("=" * 50)

train_result = trainer.train()

print("=" * 50)
print("Training completed!")


In [ ]:
print("Training Results:")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")

metrics = train_result.metrics
if "train_runtime" in metrics:
    runtime = metrics["train_runtime"]
    print(f"  Training time: {runtime:.2f}s ({runtime/60:.2f}min)")
    print(f"  Samples/second: {metrics.get('train_samples_per_second', 'N/A')}")


## 11. Evaluation


In [ ]:
eval_results = trainer.evaluate()

print("\nEvaluation Results:")
print(f"  Loss: {eval_results['eval_loss']:.4f}")
print(f"  Precision: {eval_results['eval_precision']:.4f}")
print(f"  Recall: {eval_results['eval_recall']:.4f}")
print(f"  F1-score: {eval_results['eval_f1']:.4f}")


In [ ]:
def get_detailed_report(trainer, tokenized_dataset):
    """Генерация детального отчёта по классам."""
    predictions, labels, _ = trainer.predict(tokenized_dataset["validation"])
    predictions = np.argmax(predictions, axis=2)
    
    true_predictions = []
    true_labels = []
    
    for prediction, label in zip(predictions, labels):
        true_pred = []
        true_lab = []
        
        for p, l in zip(prediction, label):
            if l != -100:
                true_pred.append(id2label[p])
                true_lab.append(id2label[l])
        
        true_predictions.append(true_pred)
        true_labels.append(true_lab)
    
    return classification_report(true_labels, true_predictions)

print("\nDetailed Classification Report:")
print("=" * 60)
print(get_detailed_report(trainer, tokenized_dataset))


## 12. Save Model


In [ ]:
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

label_config = {
    "label_list": LABEL_LIST,
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
}

with open(FINAL_MODEL_DIR / "label_config.json", "w", encoding="utf-8") as f:
    json.dump(label_config, f, ensure_ascii=False, indent=2)

print(f"\nModel saved to: {FINAL_MODEL_DIR.absolute()}")
print(f"Files saved:")
for f in FINAL_MODEL_DIR.iterdir():
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  - {f.name}: {size_mb:.2f} MB")


## 13. Test Inference


In [ ]:
from transformers import pipeline

# Примечание: для MPS используем device=-1 (CPU), так как pipeline может иметь проблемы с MPS
ner_pipeline = pipeline(
    "ner",
    model=str(FINAL_MODEL_DIR),
    tokenizer=str(FINAL_MODEL_DIR),
    aggregation_strategy="simple",  # Объединяет B- и I- теги
    device=0 if torch.cuda.is_available() else -1,
)

print("Pipeline loaded successfully!")


In [ ]:
test_texts = [
    # --- 1. Стандартные сценарии (База) ---
    "Встреча с Иваном Петровым завтра в 15:00 в переговорке",
    "Созвон в Zoom 25 декабря в 10:30 с командой разработки",
    "Напомни про день рождения мамы 20 января",

    # --- 2. Короткие фразы без глаголов (Проверка контекста) ---
    "15:35 вторник баня",
    "Завтра 19:00 спортзал",
    "Пятница вечер бар",
    "Обед в 13:00",

    # --- 3. Относительное время и предлоги ---
    "Созвон через час",
    "Напомни позвонить заказчику через 15 минут",
    "Встречаемся послезавтра в обед",
    "Синк с дизайнерами в следующий понедельник",

    # --- 4. Сленг и офисный жаргон ---
    "Кинь инвайт на дейлик на 11 утра",
    "Надо пересечься с HR завтра",
    "Брифинг по проекту SuperApp в 16:45",
    "Миток с разрабами через полчаса",

    # --- 5. Ссылки и онлайн ---
    "Вебинар тут https://meet.google.com/abc-defg в 19:00",
    "Созвон в телеграме завтра утром",

    # --- 6. Пустышки (Soft Negatives — обычная речь) ---
    "Привет, как твои дела?",
    "Какая завтра погода в Москве?",
    "Расскажи шутку",
    "Спасибо, пока",

    # --- 7. Сложные имена и группы ---
    "Встреча с Отделом Маркетинга в пятницу",
    "1:1 с Петром Ильичом завтра",
    
    # === НОВЫЕ СЕКЦИИ НИЖЕ ===

    # --- 8. Hard Negatives (Ловушки с числами и датами) ---
    # Самое важное! Тут НЕ должно быть тегов TIME или DATE.
    # Если выделит "1990" как время или "5000" как время — это ошибка.
    "Я родился в 1990 году",
    "Билет стоит 5000 рублей",
    "Мы набрали 100 очков",
    "Температура 36.6",
    "Счет в матче 2:0",  # Похоже на время, но не время

    # --- 9. Исправления и отрицания (Adversarial) ---
    # Модель часто хватает первое попавшееся слово.
    # Тут она должна понять контекст или хотя бы выделить оба варианта.
    "Давай не завтра, а в понедельник",
    "Отмени встречу в 5, перенеси на 6",
    "Нет, это было вчера",

    # --- 10. Множественные сущности (Lists) ---
    # Проверка, не "слипнутся" ли имена или даты.
    "Встреча с Сашей, Машей и Петей",
    "Нужны слоты на понедельник и среду",
    "Созвон с командой iOS и Android",

    # --- 11. Диапазоны и длительность ---
    # NER часто путается в предлогах "с" и "до".
    "Встреча с 14:00 до 16:00",
    "Забронируй переговорку на 2 часа",
    "Я буду занят до вечера"
]

print("\nTest Inference Results:")
print("=" * 60)

for text in test_texts:
    print(f"\nInput: {text}")
    results = ner_pipeline(text)
    print("Entities:")
    for entity in results:
        print(f"  - {entity['entity_group']}: '{entity['word']}' (score: {entity['score']:.3f})")
    print("-" * 40)


## Summary

### Результаты:
- Модель сохранена в `./trained_model/`
- Чекпоинты сохранены в `./checkpoints/`
- Ещё бы как-то спарсить
---



In [ ]:
import shutil
import os

# Имя папки, где лежит модель (из твоего скриншота)
source_dir = "./trained_model"
# Имя выходного архива (без расширения .zip)
output_filename = "organizer_bert_model"

# Создаем архив
shutil.make_archive(output_filename, 'zip', source_dir)

print(f"Архив {output_filename}.zip создан! Ищи его в панели справа.")